# Lab 1-01: Set Up Your Lab Environment & NVIDIA NIM API

**Chapter 1: Introduction to Generative AI and the NVIDIA Certification Path**  
**Lab Type:** Hosted-API Lab | **Estimated Time:** 30–45 minutes  
**Exam Domains Covered:** Software Development (24%), Core ML & AI Knowledge (30%)

---

## Learning Objectives
By the end of this lab, you will be able to:
1. Join the **NVIDIA Developer Program** and generate an `NVIDIA_API_KEY` on `build.nvidia.com` for hosted **NVIDIA NIM (Inference Microservices)** endpoints.
2. Verify your Python environment, PyTorch installation, and GPU hardware acceleration (`nvidia-smi`, CUDA availability, Tensor Core / BF16 support).
3. Make your first programmatic LLM call to an NVIDIA-hosted foundation model using both the **OpenAI-compatible Python SDK** and **LangChain (`langchain-nvidia-ai-endpoints`)**.
4. Inspect API response metadata (`prompt_tokens`, `completion_tokens`, latency) and implement rate-limit-aware retry/pause helpers that every subsequent lab in this course reuses.
5. Contrast **Traditional (Discriminative) AI** vs. **Generative AI** behavior on the same input text.

---
## 1. Conceptual Foundation: The NVIDIA Generative AI Stack

As covered in **Chapter 1** of the *NCA-GENL Certification Guide*, NVIDIA's Generative AI ecosystem is organized as a layered full-stack platform:

| Stack Layer | Key Technologies | Primary Role |
| :--- | :--- | :--- |
| **Application & Agent Layer** | LangChain, LlamaIndex, NeMo Guardrails, NIM Blueprints | Orchestrating RAG, multi-step agents, and safety rails |
| **Inference & Serving Layer** | **NVIDIA NIM**, **Triton Inference Server**, **TensorRT-LLM** | Packaging optimized models behind standard REST/gRPC APIs (`https://integrate.api.nvidia.com/v1`) |
| **Framework & Model Layer** | **PyTorch**, **NVIDIA NeMo Framework**, Hugging Face `transformers`, **RAPIDS (`cuDF`, `cuML`)** | Model architecture definition, distributed training, PEFT/LoRA, and GPU dataframe curation |
| **Systems & Acceleration Layer** | **CUDA**, **cuDNN**, **NCCL**, Tensor Cores | Low-level parallel compute primitives and multi-GPU collective communication |
| **Hardware Layer** | H100, H200, B200, L40S, A100, Grace Hopper (GH200), NVLink | High-bandwidth memory (HBM) and massive SIMT matrix compute |

> **EXAM TIP (Chapter 1):** Expect questions that ask you to match an NVIDIA tool to its layer in the stack:
> - **CUDA / cuDNN** $\rightarrow$ Low-level GPU compute primitives
> - **NeMo Framework** $\rightarrow$ End-to-end LLM curation, training, alignment, and PEFT
> - **TensorRT-LLM** $\rightarrow$ Inference compiler/optimizer (in-flight batching, paged KV cache, quantization)
> - **Triton Inference Server** $\rightarrow$ Multi-framework model serving engine
> - **NVIDIA NIM** $\rightarrow$ Pre-built, containerized inference microservice exposing an **OpenAI-compatible API** (`/v1/chat/completions`, `/v1/embeddings`).

---
## 2. Step-by-Step Setup: Getting Your NVIDIA API Key

All **Hosted-API labs** across Chapters 1–8 use the NVIDIA API Catalog (`build.nvidia.com`), which hosts state-of-the-art open and NVIDIA foundation models (such as `google/gemma-4-31b-it`, `nvidia/nv-embedqa-e5-v5`, and `nvidia/nv-rerankqa-mistral-4b-v3`) on NVIDIA GPU cloud infrastructure.

### How to Generate Your Free Key:
1. Navigate to [https://build.nvidia.com](https://build.nvidia.com) and sign in with (or create) a free **NVIDIA Developer Program** account.
2. Select any hosted model (for example, `google/gemma-4-31b-it`) and click **Get API Key** $\rightarrow$ **Generate Key**.
3. Copy the key immediately—it always starts with the prefix `nvapi-`.
4. Store it in an environment variable named `NVIDIA_API_KEY`.

> **Rate Limit Note (from the Lab Plan):** Hosted endpoints on the free developer tier allow roughly **40 requests per minute (RPM)**. Every lab in this course includes a lightweight rate-limit helper (`time.sleep(1.5)`) and an **automatic offline fallback mode** so that every notebook cell runs cleanly even if you are working offline or have not yet entered an API key.

In [ ]:
# Optional: Install required packages if not already present in your environment
# Uncomment the line below in Google Colab, Kaggle, or a fresh virtual environment:
# !pip install -q openai langchain-nvidia-ai-endpoints langchain-core torch pandas matplotlib

In [ ]:
import os
import sys
import time
import json
import platform
import getpass

print(f"Python Version : {sys.version.split()[0]}")
print(f"Platform       : {platform.system()} {platform.release()} ({platform.machine()})")

# Check if NVIDIA_API_KEY is already set in the environment
if not os.environ.get("NVIDIA_API_KEY"):
    print("\n[INFO] NVIDIA_API_KEY not found in environment variables.")
    print("If you have an nvapi-... key from https://build.nvidia.com, paste it below (or press Enter to run in Offline Simulation Mode):")
    try:
        entered_key = getpass.getpass("NVIDIA_API_KEY (press Enter to skip): ").strip()
        if entered_key:
            os.environ["NVIDIA_API_KEY"] = entered_key
    except Exception:
        pass

API_KEY_AVAILABLE = bool(os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"))
print(f"\nLive NVIDIA NIM Mode Enabled: {API_KEY_AVAILABLE}")
if not API_KEY_AVAILABLE:
    print("-> Running in Offline Simulation Mode (all cells will still execute and display realistic NIM outputs!).")

---
## 3. Inspecting GPU Hardware & PyTorch CUDA Readiness

Even when calling hosted NIM APIs, several labs in this course (Labs 2-02, 4-01, 5-01, and 8-02) inspect local or cloud notebook GPUs. In **Chapter 1 & Chapter 2**, the book highlights how `nvidia-smi` and `torch.cuda` report:
- **GPU Architecture & Compute Capability** (e.g., Ampere `8.0`, Ada Lovelace `8.9`, Hopper `9.0`)
- **Total & Allocated VRAM (High-Bandwidth Memory)**
- **BFloat16 (`bf16`) and Tensor Core support**

Let's run a comprehensive hardware diagnostic function that works on both GPU-equipped machines and CPU-only laptops.

In [ ]:
import os
# Prevent Windows Anaconda + PyTorch OpenMP (libiomp5md.dll) kernel crash:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import subprocess

def inspect_compute_environment():
    print("=" * 65)
    print("1. NVIDIA-SMI SYSTEM MANAGEMENT INTERFACE CHECK")
    print("=" * 65)
    try:
        smi_out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,name,driver_version,memory.total,memory.free,utilization.gpu", "--format=csv"],
            stderr=subprocess.STDOUT,
            text=True,
            timeout=5
        )
        print(smi_out.strip())
    except Exception as e:
        print(f"nvidia-smi not detected or no local NVIDIA GPU attached ({type(e).__name__}).")
        print("Tip: In Google Colab, select 'Runtime -> Change runtime type -> T4 GPU'.")

    print("\n" + "=" * 65)
    print("2. PYTORCH & CUDA DIAGNOSTICS")
    print("=" * 65)
    try:
        import torch
        print(f"PyTorch Version       : {torch.__version__}")
        cuda_ok = torch.cuda.is_available()
        print(f"CUDA Available        : {cuda_ok}")
        if cuda_ok:
            device_count = torch.cuda.device_count()
            print(f"CUDA Device Count     : {device_count}")
            print(f"CUDA Runtime Version  : {torch.version.cuda}")
            print(f"cuDNN Version         : {torch.backends.cudnn.version()}")
            for idx in range(device_count):
                props = torch.cuda.get_device_properties(idx)
                vram_gb = props.total_memory / (1024 ** 3)
                print(f"  [GPU {idx}] {props.name} | VRAM: {vram_gb:.2f} GB | "
                      f"Compute Capability: {props.major}.{props.minor} | "
                      f"SM Count: {props.multi_processor_count}")
            print(f"BF16 Supported        : {torch.cuda.is_bf16_supported()}")
        else:
            print("Active Device         : CPU (Fallback ready for all labs)")
    except ImportError:
        print("PyTorch is not installed in this environment.")

inspect_compute_environment()

---
## 4. Calling Your First Hosted LLM via the OpenAI-Compatible NIM Client

One of NVIDIA NIM's core design principles (**Chapter 1 & Chapter 3**) is **drop-in API compatibility**. Because NIM exposes standard `/v1/chat/completions` and `/v1/embeddings` endpoints, any application written with the standard `openai` Python SDK can switch to an NVIDIA-hosted or self-hosted NIM container simply by changing:
- `base_url="https://integrate.api.nvidia.com/v1"` (or `http://localhost:8000/v1` for a local NIM container)
- `api_key=os.environ["NVIDIA_API_KEY"]`
- `model="google/gemma-4-31b-it"`

> **Important Note on Reasoning / Thinking Models (such as `google/gemma-4-31b-it`):**  
> On NVIDIA NIM, reasoning-capable models generate internal thinking tokens (`delta.reasoning_content`) before emitting their final answer (`delta.content`). Setting `max_tokens` too low (e.g., `150`) causes the model to exhaust its token budget during the thinking phase and return `message.content = None`. Our `call_nim_chat()` helper below uses **streaming (`stream=True`)** with an automatic minimum token budget (`max(max_tokens, 1024)`) and captures both `content` and `reasoning_content` so every model on `build.nvidia.com` works reliably!

In [ ]:
NIM_BASE_URL = "https://integrate.api.nvidia.com/v1"
DEFAULT_MODEL = "google/gemma-4-31b-it"


def _stream_nim_completion(client, model: str, messages: list, temperature: float, max_tokens: int) -> tuple:
    '''
    Streams chat completion chunks from NVIDIA NIM (matching build.nvidia.com's official streaming pattern).
    Separately collects delta.content and delta.reasoning_content so reasoning models (like google/gemma-4-31b-it)
    never return None or hang waiting for non-streaming completion.
    '''
    effective_max_tokens = max(max_tokens, 1024)
    content_pieces = []
    reasoning_pieces = []
    finish_reason = "stop"

    try:
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            top_p=0.9,
            max_tokens=effective_max_tokens,
            stream=True,
        )
    except Exception as first_err:
        # Some Gemma chat templates require system instructions to be merged into the first user message
        if any(m.get("role") == "system" for m in messages):
            sys_text = "\n".join(m["content"] for m in messages if m.get("role") == "system")
            user_msgs = [m for m in messages if m.get("role") != "system"]
            merged_messages = []
            for i, m in enumerate(user_msgs):
                if i == 0:
                    merged_messages.append({"role": "user", "content": f"{sys_text}\n\n{m['content']}"})
                else:
                    merged_messages.append(m)
            stream = client.chat.completions.create(
                model=model,
                messages=merged_messages,
                temperature=temperature,
                top_p=0.9,
                max_tokens=effective_max_tokens,
                stream=True,
            )
        else:
            raise first_err

    for chunk in stream:
        if not getattr(chunk, "choices", None):
            continue
        choice0 = chunk.choices[0]
        if getattr(choice0, "finish_reason", None):
            finish_reason = choice0.finish_reason
        delta = getattr(choice0, "delta", None)
        if delta is None:
            continue
        r_piece = getattr(delta, "reasoning_content", None)
        if not r_piece and hasattr(delta, "model_extra") and isinstance(delta.model_extra, dict):
            r_piece = delta.model_extra.get("reasoning_content")
        if r_piece:
            reasoning_pieces.append(str(r_piece))

        c_piece = getattr(delta, "content", None)
        if c_piece is not None:
            content_pieces.append(str(c_piece))

    answer_text = "".join(content_pieces).strip()
    reasoning_text = "".join(reasoning_pieces).strip()
    final_content = answer_text if answer_text else reasoning_text
    return final_content, reasoning_text, finish_reason


def call_nim_chat(
    messages: list,
    model: str = DEFAULT_MODEL,
    temperature: float = 0.2,
    max_tokens: int = 512,
    pause_seconds: float = 1.5,
) -> dict:
    '''
    Calls the NVIDIA NIM chat completions endpoint using the OpenAI-compatible SDK.
    Automatically falls back to a deterministic simulated response if NVIDIA_API_KEY is not set
    or if the endpoint is unreachable.
    '''
    t0 = time.perf_counter()

    # Prepare deterministic fallback reply so content is NEVER None
    all_prompt_text = " ".join(m["content"] for m in messages)
    simulated_reply = (
        "NVIDIA NIM (NVIDIA Inference Microservices) is a set of pre-built, containerized microservices "
        "that accelerate the deployment of foundation models across cloud, data center, and workstations. "
        "Under the hood, NIM automatically selects optimized execution engines like TensorRT-LLM and serves "
        "models through industry-standard OpenAI-compatible REST APIs."
    )
    if "JSON" in all_prompt_text or "json" in all_prompt_text:
        simulated_reply = json.dumps({
            "sentiment": "negative",
            "product": "Cloud GPU Instance",
            "priority": "high",
            "draft_reply": "We apologize for the CUDA driver mismatch on your cluster node. Our engineering team has provisioned a replacement H100 instance and credited your account."
        }, indent=2)
    elif "10-year-old" in all_prompt_text:
        simulated_reply = (
            "Think of a CPU like a super-smart head chef who cooks one fancy dish at a time, while a GPU is like "
            "thousands of kitchen helpers chopping vegetables all at the exact same second! Large Language Models need "
            "to multiply millions of numbers at once, so the thousands of helpers inside a GPU finish the job way faster."
        )
    elif "Senior GPU Systems Engineer" in all_prompt_text:
        simulated_reply = (
            "Transformer prefill and decode stages are dominated by dense GEMM operations and high-bandwidth KV-cache "
            "reads. GPUs outperform CPUs via massive SIMT thread parallelism, specialized Tensor Cores (FP16/BF16/FP8 MMA), "
            "and multi-TB/s HBM memory bandwidth that alleviates the memory-bandwidth bottleneck during autoregressive decoding."
        )

    if os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
        try:
            from openai import OpenAI

            print(f"[NIM API] Streaming response from '{model}'...", flush=True)
            client = OpenAI(
                base_url=NIM_BASE_URL,
                api_key=os.environ["NVIDIA_API_KEY"],
                timeout=60.0,
            )
            content, reasoning_text, finish_reason = _stream_nim_completion(
                client, model, messages, temperature, max_tokens
            )
            elapsed_ms = (time.perf_counter() - t0) * 1000.0
            time.sleep(pause_seconds)  # Respect ~40 RPM developer tier rate limit

            safe_content = str(content) if content else simulated_reply
            p_toks = max(12, len(all_prompt_text.split()) * 4 // 3)
            r_toks = len(reasoning_text.split()) * 4 // 3 if reasoning_text else 0
            c_toks = max(1, len(safe_content.split()) * 4 // 3) + r_toks

            return {
                "mode": "LIVE_NIM_API" if content else "LIVE_NIM_API (Fallback Content)",
                "model": model,
                "content": safe_content,
                "reasoning_tokens_est": r_toks,
                "finish_reason": finish_reason or "stop",
                "prompt_tokens": p_toks,
                "completion_tokens": c_toks,
                "total_tokens": p_toks + c_toks,
                "latency_ms": round(elapsed_ms, 1),
            }
        except Exception as api_err:
            print(f"[WARN] Live NIM API call encountered {type(api_err).__name__}: {api_err}. Using offline fallback.")

    elapsed_ms = (time.perf_counter() - t0) * 1000.0 + 185.0
    prompt_toks = max(12, len(all_prompt_text.split()) * 4 // 3)
    comp_toks = max(15, len(simulated_reply.split()) * 4 // 3)
    return {
        "mode": "OFFLINE_SIMULATION",
        "model": model,
        "content": str(simulated_reply),
        "reasoning_tokens_est": 0,
        "finish_reason": "stop",
        "prompt_tokens": prompt_toks,
        "completion_tokens": comp_toks,
        "total_tokens": prompt_toks + comp_toks,
        "latency_ms": round(elapsed_ms, 1),
    }


# Test our first NIM call!
test_messages = [
    {"role": "system", "content": "You are a concise NVIDIA AI solutions architect."},
    {"role": "user", "content": "In 2-3 sentences, explain what NVIDIA NIM is and how it relates to TensorRT-LLM."}
]

result = call_nim_chat(test_messages, temperature=0.2, max_tokens=512)
print(f"Execution Mode    : {result['mode']}")
print(f"Model             : {result['model']}")
print(f"Finish Reason     : {result['finish_reason']}")
print(f"Token Usage (Est) : {result['prompt_tokens']} prompt + {result['completion_tokens']} completion = {result['total_tokens']} total tokens")
print(f"Latency           : {result['latency_ms']} ms")
print("-" * 65)
print(f"Model Response:\n{result['content']}")

---
## 5. Calling NVIDIA Endpoints with LangChain (`ChatNVIDIA`)

In **Chapter 3**, you will build multi-step Retrieval-Augmented Generation (RAG) pipelines using **LangChain** and the official `langchain-nvidia-ai-endpoints` package (`ChatNVIDIA`, `NVIDIAEmbeddings`, and `NVIDIARerank`). Let's verify how `ChatNVIDIA` wraps the exact same `DEFAULT_MODEL` endpoint in LangChain's `Runnable` interface.

In [ ]:
def test_langchain_nvidia():
    if os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
        try:
            from langchain_nvidia_ai_endpoints import ChatNVIDIA
            llm = ChatNVIDIA(model=DEFAULT_MODEL, temperature=0.1, max_tokens=1024)
            resp = llm.invoke("List the 5 exam domains of the NVIDIA NCA-GENL certification in one bulleted list.")
            content = getattr(resp, "content", None)
            if not content and hasattr(resp, "additional_kwargs"):
                content = resp.additional_kwargs.get("reasoning_content")
            if content:
                print(f"[LangChain ChatNVIDIA Live Output ({DEFAULT_MODEL})]")
                print(content)
                return
        except Exception as e:
            print(f"[INFO] langchain-nvidia-ai-endpoints fallback ({type(e).__name__}); showing reference output.")

    print("[LangChain ChatNVIDIA Reference Output]")
    print(
        "• Core Machine Learning & AI Knowledge (30%)\n"
        "• Software Development (24%)\n"
        "• Experimentation (22%)\n"
        "• Data Analysis & Visualization (14%)\n"
        "• Trustworthy AI (10%)"
    )

test_langchain_nvidia()

---
## 6. Traditional (Discriminative) AI vs. Generative AI in Practice

In **Chapter 1 (*What Is Generative AI?*)**, the book contrasts two foundational paradigms:
1. **Discriminative / Predictive AI ($P(y \mid x)$):** Maps an input $x$ to a fixed label or numerical value from a predefined set (e.g., classifying a support ticket as `Positive`/`Negative`, or predicting a churn score). It **cannot** synthesize novel text.
2. **Generative AI ($P(x)$ or $P(y_{\text{seq}} \mid x)$):** Models the underlying distribution of data to generate brand-new content—AND can also perform discriminative tasks via natural-language prompting (**in-context learning**).

Let's demonstrate how a single LLM call can perform **both** discriminative classification (sentiment + priority routing) and generative synthesis (drafting a personalized customer response) in a single structured JSON payload.

In [ ]:
support_ticket = (
    "Ticket #4092: Our nightly training job failed after upgrading the node image. "
    "PyTorch reports a CUDA driver mismatch on the 8x H100 cluster and our deadline is tomorrow morning!"
)

hybrid_messages = [
    {
        "role": "system",
        "content": (
            "You are an enterprise support assistant. Analyze the customer ticket and return ONLY a valid JSON object "
            "with keys: 'sentiment' ('positive'|'neutral'|'negative'), 'product' (string), 'priority' ('low'|'medium'|'high'), "
            "and 'draft_reply' (a helpful 2-sentence customer response)."
        ),
    },
    {"role": "user", "content": support_ticket},
]

hybrid_result = call_nim_chat(hybrid_messages, temperature=0.1, max_tokens=512)
print("Input Ticket:")
print(" ", support_ticket)
print("\nCombined Discriminative + Generative Output (JSON):")
print(hybrid_result["content"])

---
## 7. Hands-On Exercises

Complete the following exercises to solidify your setup and Chapter 1 knowledge.

### Exercise 1: System Prompt Persona & Temperature Comparison
Write a function `compare_personas(question: str)` that calls `call_nim_chat` twice on the same user question (`"Why do Large Language Models need GPUs instead of CPUs?"`):
1. Once with a **System Prompt** instructing the model to explain to a **10-year-old student** (`temperature=0.7`).
2. Once with a **System Prompt** instructing the model to explain to a **Senior GPU Systems Engineer** (`temperature=0.1`).
Print both responses and compare their token usage.

In [ ]:
# TODO: Complete Exercise 1
def compare_personas(question: str):
    # 1. Call call_nim_chat with a 10-year-old persona and temperature=0.7
    # 2. Call call_nim_chat with a Senior GPU Systems Engineer persona and temperature=0.1
    # 3. Print both outputs and their token counts
    pass

# Uncomment to test:
# compare_personas("Why do Large Language Models need GPUs instead of CPUs?")

### Exercise 2: Environment Readiness Checklist Validator
Write a Python function `generate_lab_readiness_report()` that returns a dictionary summarizing your machine's readiness for all 21 labs:
- `python_ok`: `True` if Python version is $\ge 3.9$
- `api_key_configured`: `True` if `NVIDIA_API_KEY` starts with `"nvapi-"`
- `gpu_available`: `True` if `torch.cuda.is_available()` is `True`
- `recommended_execution_tier`: `"Local GPU + Hosted NIM"`, `"Hosted NIM + Cloud GPU (Colab/Kaggle) for GPU Labs"`, or `"Offline Simulation Mode"`

In [ ]:
# TODO: Complete Exercise 2
def generate_lab_readiness_report() -> dict:
    # Write your readiness checks here
    pass

# Uncomment to test:
# print(json.dumps(generate_lab_readiness_report(), indent=2))

---
## 8. Reference Solutions

In [ ]:
import os
# Prevent Windows Anaconda + PyTorch OpenMP (libiomp5md.dll) kernel crash:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Solution to Exercise 1
def compare_personas_solution(question: str):
    personas = [
        ("10-Year-Old Student", "Explain concepts simply for a curious 10-year-old using everyday analogies in 3 sentences.", 0.7),
        ("Senior GPU Engineer", "Explain concepts rigorously for a Senior GPU Systems Engineer using terms like SIMT, HBM bandwidth, and Tensor Cores in 3 sentences.", 0.1),
    ]
    for label, sys_prompt, temp in personas:
        res = call_nim_chat(
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": question},
            ],
            temperature=temp,
            max_tokens=512,
        )
        print(f"=== Persona: {label} (temp={temp}, tokens={res['total_tokens']}) ===")
        print(res["content"] + "\n")

compare_personas_solution("Why do Large Language Models need GPUs instead of CPUs?")

# Solution to Exercise 2
def generate_lab_readiness_report_solution() -> dict:
    py_ok = sys.version_info >= (3, 9)
    key_ok = os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-")
    try:
        import torch
        gpu_ok = bool(torch.cuda.is_available())
    except ImportError:
        gpu_ok = False

    if gpu_ok and key_ok:
        tier = "Local GPU + Hosted NIM"
    elif key_ok:
        tier = "Hosted NIM + Cloud GPU (Colab/Kaggle) for GPU Labs"
    else:
        tier = "Offline Simulation Mode (Ready for all 21 notebooks)"

    return {
        "python_ok": py_ok,
        "api_key_configured": key_ok,
        "gpu_available": gpu_ok,
        "recommended_execution_tier": tier,
    }

print("Readiness Report:")
print(json.dumps(generate_lab_readiness_report_solution(), indent=2))

---
## 9. NCA-GENL Chapter 1 Review Takeaways

- **Discriminative vs. Generative AI:** Discriminative models learn decision boundaries $P(y \mid x)$; Generative models learn the data distribution to synthesize new text, code, images, or audio.
- **NVIDIA AI Stack:** `CUDA` (parallel compute platform) $\rightarrow$ `NeMo` (LLM training/alignment/curation framework) $\rightarrow$ `TensorRT-LLM` (inference optimization library) $\rightarrow$ `Triton` (model serving server) $\rightarrow$ `NIM` (pre-packaged containerized inference microservice with OpenAI-compatible APIs).
- **Rate Limits:** Always pace requests to `https://integrate.api.nvidia.com/v1` on the free Developer tier (~40 RPM).